In [1]:
# imports
from pathlib import Path
import pandas as pd
import geopandas as gpd
import platform
import seaborn as sns
from sklearn.linear_model import LinearRegression
from IPython.display import display, clear_output
import ipywidgets as widgets
import matplotlib.pyplot as plt




In [2]:
# establish paths to datasets
project_fldr = Path.cwd().parents[1]
raw_fldr = project_fldr / 'data' / 'raw'
processed_fldr = project_fldr / 'data' / 'processed'
figure_fldr = project_fldr/"figures"

# set a path to the main box folder based on machine's os
# if windows: 
if platform.system() == 'Windows':
    box_path = Path.home() / 'Box'
# if mac:
else:
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

#Confirm that the box path is valid
if box_path.exists():
    print(f'Box folder found at {box_path}')
else:
    print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

wetlands_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'wetlands_subset' / 'wetlands_subset.geojson'
simpler_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'simpler_wetlands'


Box folder found at /Users/masonbunker/Library/CloudStorage/Box-Box


# read in the data to be used

In [3]:
n10000 = gpd.read_file(processed_fldr/"n10000_random_pts.geojson")

n10000

DataSourceError: /Users/masonbunker/Desktop/Duke/Environmental Data Exploration/Wildfire-Wetland_BunkerAndersonTseng/data/processed/n10000_random_pts.geojson: No such file or directory

this is ### out because the above code simplifies it

In [ ]:
# # CA outline
# ca_outline = gpd.read_file(simpler_path/"ca_outline.geojson")

# ca_outline = ca_outline.to_crs(epsg=26910)

# ca_outline.crs

<Projected CRS: EPSG:26910>
Name: NAD83 / UTM zone 10N
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: North America - between 126°W and 120°W - onshore and offshore. Canada - British Columbia; Northwest Territories; Yukon. United States (USA) - California; Oregon; Washington.
- bounds: (-126.0, 30.54, -119.99, 81.8)
Coordinate Operation:
- name: UTM zone 10N
- method: Transverse Mercator
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [ ]:
# # fires

# # pull in the fires into dfs from geojsons
# fires = ["fire_2020", "fire_2021", "fire_2022", "fire_2023", "fire_2024", "fire_2025"]
# years=[2020, 2021, 2022, 2023, 2024, 2025]

# for fire, year in zip(fires, years):
#     globals()[fire] = gpd.read_file(
#         processed_fldr/ "fire_explore" / f"fire_{year}.geojson"
#     )

# fire_dfs = [fire_2020, fire_2021, fire_2022, fire_2023, fire_2024, fire_2025]

# fire_all = pd.concat(fire_dfs)

# fire_all = fire_all.to_crs(epsg=26910)

In [6]:
# read in the wetlands geojson file, this will take a WHILE - 10 mins 1st try 
wetlands = gpd.read_file(wetlands_path, engine="pyogrio")

In [7]:
wetlands = wetlands.to_crs(epsg=26910)

# generate the random points in a geodataframe

In [ ]:
# def sample_distance(size):
#     # create the sample points to start this will be a geoseries
#     globals()[f"n{size}"] = ca_outline.geometry.sample_points(size=size, rng=42) #rng sets the same sample seed every time

#     # turn the geoseries into a geodataframe
#     globals()[f"n{size}"] = gpd.GeoDataFrame(
#         geometry=globals()[f"n{size}"].explode(index_parts=False),
#         crs=ca_outline.crs
#     )

#     #reset the index (for some reason wouldnt work above)
#     globals()[f"n{size}"] = globals()[f"n{size}"].reset_index(drop=True)

#     # insert the columns for next analysis steps
#     cols = ["Wet_dist", "Fire_dist"]
#     globals()[f"n{size}"][cols]=None

#     # NOW ADD IN THE DISTANCE ASPECT
#     # nearest fire
#     globals()[f"n{size}"] = gpd.sjoin_nearest(
#          globals()[f"n{size}"],
#         fire_all,
#         how="left",
#         distance_col="Fire_dist"
#     )

#     # drop the index because otherwise both joins cant happen
#     globals()[f"n{size}"] =  globals()[f"n{size}"].drop(columns=["index_right"])


#     # nearest wetland
#     globals()[f"n{size}"] = gpd.sjoin_nearest(
#          globals()[f"n{size}"],
#         wetlands,
#         how="left",
#         distance_col="Wet_dist"
#     )

#     globals()[f"n{size}"] =  globals()[f"n{size}"].drop(columns=["index_right"])

#     print(f"Sample DataFrame Name: n{size}")
    

# Regressions time

In [ ]:
# sample_distance(100000)

Sample DataFrame Name: n100000


In [ ]:
system_colors = {
    "Marine": "#02518a",
    "Estuarine": "#0b2ce9",
    "Riverine": "#17becf",
    "Lacustrine": "#01f694",
    "Palustrine": "#4bcb00"
}

dist_plot = sns.lmplot(
    data = n10000,
    x = "Wet_dist",
    y = "Fire_dist",
    hue = "C Code",
    #palette=system_colors,
    height=10,
)

#Set plot title and axis labels
dist_plot.set_axis_labels("Distance to Nearest Wetland", "Distance to Nearest Fire"),
dist_plot.set(title="10,000 random points in California\ndistance to wetlands vs fire")

#Clean up the legend
sns.move_legend(
    dist_plot, 
    loc = 'upper right',
    bbox_to_anchor=(0.8, 0.95)
)

# Display
dist_plot

# DEVELOP THE WIDGETS

In [ ]:
n10000.head()

In [ ]:
# plot as a function

def plot_points(
        System = "Riverine",
        Subsystem = "Intermittent",
        Class = "Streambed",
        Code = "R4SBA"):
    
    # Subset the data
    df_subset = n10000.loc[
        (n10000["C System"]==System) &
        (n10000["C Subsystem"]==Subsystem) &
        (n10000["C Class"]==Class) &
        (n10000["C Code"].isin(Code)),
    ]

    # Create a plot of the selected points
    dist_plot = sns.lmplot(
        data = n10000,
        x = "Wet_dist",
        y = "Fire_dist",
        hue = "C Code",
        #palette=system_colors,
        height=10,
    )

    #Set plot title and axis labels
    dist_plot.set_axis_labels("Distance to Nearest Wetland", "Distance to Nearest Fire"),
    dist_plot.set(title="10,000 random points in California\nDistance to Wetlands vs Fire")

    #Clean up the legend
    sns.move_legend(
        dist_plot, 
        loc = 'upper right',
        bbox_to_anchor=(0.8, 0.95)
    )

    # Return Display
    return dist_plot


## Widget time

In [ ]:
# drop down for C system
system_list = n10000["C System"].unique()
system_dropdown = widgets.Dropdown(
    options=system_list,
    value=system_list[0],
    description="System: ",
)



In [ ]:
# drop down for C subsystem

subsystem_list = n10000["C Subsystem"].unique()
subsystem_dropdown = widgets.Dropdown(
    options=subsystem_list,
    value=subsystem_list[0],
    description="Subsystem: ",
)


In [ ]:
# dropdown for C class
class_list = n10000["C Class"].unique()
class_dropdown = widgets.Dropdown(
    options=class_list,
    value=class_list[0],
    description="Class: ",
)

In [ ]:
# checkbox for C Code
codes = n10000["C Code"].unique()
codes_select = widgets.SelectMultiple(
    options = codes,
    value=(codes[0],),
    description="Code: ",
)


## Add reactivity

In [ ]:
# Create an output area for the plot
plot_output = widgets.Output()

# Function to update the plot
def update_data_or_plot(change=None):
    #Direct output to the plot_output widget
    with plot_output:
        # Clear the previous plot
        clear_output(wait=True)

        # Generate the new plot
        g = plot_points(
            System=system_dropdown.value,
            Subsystem=subsystem_dropdown.value,
            Class=class_dropdown.value,
            Code=codes_select.value
        )

        # Explicitly display the plot
        display(g.figure)
        plt.close(g.figure)

# Attach observers
system_dropdown.observe(update_data_or_plot, names='value')
subsystem_dropdown.observe(update_data_or_plot, names='value')
class_dropdown.observe(update_data_or_plot, names='value')
codes_select.observe(update_data_or_plot, names='value')

# Display the widgets and plot together
display(
    widgets.VBox([
        system_dropdown,
        subsystem_dropdown,
        class_dropdown,
        codes_select,
        plot_output
    ])
)

# Generate the initial plot
update_data_or_plot()

In [ ]:
# # save n10000 because it will take less time and less data to read in
# n100000.to_file(
#     processed_fldr/"n100000_random_pts.geojson", driver= "GeoJSON"
# )


In [ ]:
# generate 1million random points so more in each category?
sample_distance(5000000)